# GPT-2 Model Testing

This notebook demonstrates:
1. Loading training configuration from `configs/training/trigo-gpt2.yaml`
2. Creating TGN dataset and dataloader
3. Initializing GPT-2 model with AttentionCausalLoss wrapper
4. Forward pass through model with one batch
5. Backward pass (loss backpropagation)
6. Inspecting gradients and model outputs

## 1. Environment Setup

In [3]:
import sys
import os


sys.path.append('..')
os.chdir('..')

In [5]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from omegaconf import OmegaConf

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

from trigor.data import TGNDataset
from trigor.models import make_model

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Project root: /home/camus/work/trigoRL
PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3090


## 2. Load Configuration

Load the GPT-2 training configuration from YAML file.

In [6]:
# Load configuration
config_path = project_root / "configs/training/trigo-gpt2.yaml"
cfg = OmegaConf.load(config_path)

# Resolve paths
OmegaConf.update(cfg, "paths.root", str(project_root))
OmegaConf.resolve(cfg)

print("Configuration loaded successfully!")
print("\nModel configuration:")
print(f"  Type: {cfg.model.type}")
print(f"  Base model: {cfg.model.config.model_config.type}")
print(f"  Hidden size: {cfg.model.config.model_config.config.hidden_size}")
print(f"  Num layers: {cfg.model.config.model_config.config.num_layers}")
print(f"  Num heads: {cfg.model.config.model_config.config.num_heads}")
print(f"  Vocab size: {cfg.model.config.model_config.config.vocab_size}")
print(f"  Max seq len: {cfg.model.config.model_config.config.max_seq_len}")
print(f"\n  Ignore index: {cfg.model.config.ignore_index}")
print(f"  Label smoothing: {cfg.model.config.label_smoothing}")

print("\nData configuration:")
print(f"  Data dir: {cfg.data.data_dir}")
print(f"  Max length: {cfg.data.max_length}")
print(f"  Batch size: {cfg.data.loader.batch_size}")

print("\nTraining configuration:")
print(f"  Learning rate: {cfg.training.learning_rate}")
print(f"  Weight decay: {cfg.training.weight_decay}")
print(f"  Max grad norm: {cfg.training.max_grad_norm}")

Configuration loaded successfully!

Model configuration:
  Type: AttentionCausalLoss
  Base model: GPT2CausalLM
  Hidden size: 256
  Num layers: 6
  Num heads: 8
  Vocab size: 259
  Max seq len: 2048

  Ignore index: 256
  Label smoothing: 0.1

Data configuration:
  Data dir: /home/camus/work/trigoRL/third_party/trigo/trigo-web/tools/output
  Max length: 2048
  Batch size: 8

Training configuration:
  Learning rate: 0.0001
  Weight decay: 0.01
  Max grad norm: 1.0


## 3. Load Dataset

Create TGNDataset and inspect some samples.

In [7]:
# Create dataset
print("Loading TGN dataset...")
dataset = TGNDataset.from_config(cfg.data)

print(f"\nDataset loaded successfully!")
print(f"  Total samples: {len(dataset)}")
print(f"  Tokenizer vocab size: {dataset.tokenizer.get_vocab_size()}")
print(f"  Max length: {dataset.max_length}")
print(f"  PAD token ID: {dataset.tokenizer.PAD_TOKEN_ID}")
print(f"  START token ID: {dataset.tokenizer.START_TOKEN_ID}")
print(f"  END token ID: {dataset.tokenizer.END_TOKEN_ID}")

Loading TGN dataset...
Loaded 100 TGN files from /home/camus/work/trigoRL/third_party/trigo/trigo-web/tools/output

Dataset loaded successfully!
  Total samples: 100
  Tokenizer vocab size: 259
  Max length: 2048
  PAD token ID: 256
  START token ID: 257
  END token ID: 258


In [8]:
# Inspect a single sample
print("\n" + "="*80)
print("Sample Inspection")
print("="*80)

sample = dataset[0]
print(f"\nSample 0:")
print(f"  Input IDs shape: {sample['input_ids'].shape}")
print(f"  Labels shape: {sample['labels'].shape}")
print(f"  Attention mask shape: {sample['attention_mask'].shape}")

print(f"\n  First 20 input tokens: {sample['input_ids'][:20].tolist()}")
print(f"  First 20 labels: {sample['labels'][:20].tolist()}")
print(f"  First 20 attention mask: {sample['attention_mask'][:20].tolist()}")

# Count non-padding tokens
non_pad_count = (sample['attention_mask'] == 1).sum().item()
print(f"\n  Non-padding tokens: {non_pad_count} / {sample['input_ids'].shape[0]}")


Sample Inspection

Sample 0:
  Input IDs shape: torch.Size([2047])
  Labels shape: torch.Size([2047])
  Attention mask shape: torch.Size([2047])

  First 20 input tokens: [257, 91, 66, 111, 97, 114, 100, 32, 51, 120, 52, 120, 52, 93, 10, 10, 49, 46, 32, 48]
  First 20 labels: [91, 66, 111, 97, 114, 100, 32, 51, 120, 52, 120, 52, 93, 10, 10, 49, 46, 32, 48, 121]
  First 20 attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

  Non-padding tokens: 304 / 2047


## 4. Create DataLoader

Create dataloader with collate function for batching.

In [9]:
# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=cfg.data.loader.batch_size,
    shuffle=cfg.data.loader.shuffle,
    num_workers=0,  # Use 0 for notebook to avoid multiprocessing issues
    collate_fn=TGNDataset.collate_batch,
    pin_memory=False,  # Disable for CPU testing
)

print(f"DataLoader created successfully!")
print(f"  Batch size: {cfg.data.loader.batch_size}")
print(f"  Total batches: {len(dataloader)}")
print(f"  Shuffle: {cfg.data.loader.shuffle}")

DataLoader created successfully!
  Batch size: 8
  Total batches: 13
  Shuffle: True


In [10]:
# Get one batch for testing
batch = next(iter(dataloader))

print("\n" + "="*80)
print("Batch Inspection")
print("="*80)

print(f"\nBatch shapes:")
print(f"  Input IDs: {batch['input_ids'].shape}")
print(f"  Labels: {batch['labels'].shape}")
print(f"  Attention mask: {batch['attention_mask'].shape}")

print(f"\nBatch statistics:")
print(f"  Total tokens: {batch['input_ids'].numel()}")
print(f"  Valid tokens: {(batch['attention_mask'] == 1).sum().item()}")
print(f"  Padding tokens: {(batch['attention_mask'] == 0).sum().item()}")
print(f"  Padding ratio: {(batch['attention_mask'] == 0).sum().item() / batch['input_ids'].numel():.2%}")

# Show first sample in batch
print(f"\nFirst sample in batch:")
print(f"  First 20 tokens: {batch['input_ids'][0, :20].tolist()}")
print(f"  Valid length: {(batch['attention_mask'][0] == 1).sum().item()}")


Batch Inspection

Batch shapes:
  Input IDs: torch.Size([8, 2047])
  Labels: torch.Size([8, 2047])
  Attention mask: torch.Size([8, 2047])

Batch statistics:
  Total tokens: 16376
  Valid tokens: 10767
  Padding tokens: 5609
  Padding ratio: 34.25%

First sample in batch:
  First 20 tokens: [257, 91, 66, 111, 97, 114, 100, 32, 57, 120, 50, 120, 57, 93, 10, 10, 49, 46, 32, 98]
  Valid length: 1097


## 5. Create Model

Initialize GPT-2 model with AttentionCausalLoss wrapper.

In [11]:
# Create model using factory
print("Creating model...")
model = make_model(cfg.model.type, cfg.model.config)

print("\nModel created successfully!")
print(model)

# Count parameters
params = model.count_parameters()
print(f"\nParameter counts:")
print(f"  Total: {params['total']:,}")
print(f"  Trainable: {params['trainable']:,}")
print(f"  Non-trainable: {params['non_trainable']:,}")

# Get model info
info = model.get_model_info()
print(f"\nModel info:")
for key, value in info.items():
    print(f"  {key}: {value}")

Creating model...

Model created successfully!
AttentionCausalLoss(
  model_type=GPT2CausalLM,
  parameters=5,329,664,
  ignore_index=256,
  label_smoothing=0.1
)

Parameter counts:
  Total: 5,329,664
  Trainable: 5,329,664
  Non-trainable: 0

Model info:
  model_type: GPT2CausalLM
  ignore_index: 256
  label_smoothing: 0.1
  model_info: {'model_type': 'gpt2', 'vocab_size': 259, 'hidden_size': 256, 'num_layers': 6, 'num_heads': 8, 'max_seq_len': 2048, 'intermediate_size': 1024, 'activation': 'gelu_new', 'dropout': 0.1, 'total_parameters': 5329664, 'trainable_parameters': 5329664}


## 6. Forward Pass

Run forward pass through the model with one batch.

In [12]:
# Set model to training mode
model.train()

print("\n" + "="*80)
print("Forward Pass")
print("="*80)

# Forward pass
print("\nRunning forward pass...")
outputs = model(
    input_ids=batch['input_ids'],
    labels=batch['labels'],
    attention_mask=batch['attention_mask'],
    return_logits=True,  # Return logits for inspection
)

print("\nForward pass completed!")
print(f"\nOutput keys: {list(outputs.keys())}")

print(f"\nOutput shapes:")
print(f"  Loss: {outputs['loss'].shape} (scalar)")
print(f"  Accuracy: {outputs['accuracy'].shape} (scalar)")
print(f"  Top-5 Accuracy: {outputs['top5_accuracy'].shape} (scalar)")
print(f"  Perplexity: {outputs['perplexity'].shape} (scalar)")
print(f"  Num tokens: {outputs['num_tokens'].shape} (scalar)")
print(f"  Logits: {outputs['logits'].shape} [batch_size, seq_len, vocab_size]")

print(f"\nMetrics:")
print(f"  Loss: {outputs['loss'].item():.4f}")
print(f"  Accuracy: {outputs['accuracy'].item():.4f} ({outputs['accuracy'].item()*100:.2f}%)")
print(f"  Top-5 Accuracy: {outputs['top5_accuracy'].item():.4f} ({outputs['top5_accuracy'].item()*100:.2f}%)")
print(f"  Perplexity: {outputs['perplexity'].item():.2f}")
print(f"  Valid tokens: {outputs['num_tokens'].item()}")

# Analyze logits
logits = outputs['logits']
print(f"\nLogits statistics:")
print(f"  Min: {logits.min().item():.4f}")
print(f"  Max: {logits.max().item():.4f}")
print(f"  Mean: {logits.mean().item():.4f}")
print(f"  Std: {logits.std().item():.4f}")

# Get predictions
predictions = torch.argmax(logits, dim=-1)
print(f"\nPredictions shape: {predictions.shape}")
print(f"First sample predictions (first 20): {predictions[0, :20].tolist()}")
print(f"First sample labels (first 20): {batch['labels'][0, :20].tolist()}")


Forward Pass

Running forward pass...

Forward pass completed!

Output keys: ['loss', 'accuracy', 'perplexity', 'top5_accuracy', 'num_tokens', 'logits']

Output shapes:
  Loss: torch.Size([]) (scalar)
  Accuracy: torch.Size([]) (scalar)
  Top-5 Accuracy: torch.Size([]) (scalar)
  Perplexity: torch.Size([]) (scalar)
  Num tokens: torch.Size([]) (scalar)
  Logits: torch.Size([8, 2047, 259]) [batch_size, seq_len, vocab_size]

Metrics:
  Loss: 5.5731
  Accuracy: 0.0348 (3.48%)
  Top-5 Accuracy: 0.0550 (5.50%)
  Perplexity: 263.26
  Valid tokens: 10762

Logits statistics:
  Min: -1.6351
  Max: 2.2447
  Mean: 0.0096
  Std: 0.3308

Predictions shape: torch.Size([8, 2047])
First sample predictions (first 20): [226, 91, 66, 111, 97, 114, 100, 32, 74, 120, 50, 31, 100, 93, 10, 10, 170, 46, 32, 90]
First sample labels (first 20): [91, 66, 111, 97, 114, 100, 32, 57, 120, 50, 120, 57, 93, 10, 10, 49, 46, 32, 98, 97]


## 7. Backward Pass

Perform backward pass and inspect gradients.

In [13]:
print("\n" + "="*80)
print("Backward Pass")
print("="*80)

# Get loss
loss = outputs['loss']
print(f"\nLoss before backward: {loss.item():.4f}")
print(f"Loss requires_grad: {loss.requires_grad}")

# Zero gradients (simulating optimizer.zero_grad())
model.zero_grad()
print("\nGradients zeroed.")

# Backward pass
print("\nRunning backward pass...")
loss.backward()
print("Backward pass completed!")


Backward Pass

Loss before backward: 5.5731
Loss requires_grad: True

Gradients zeroed.

Running backward pass...
Backward pass completed!


## 8. Inspect Gradients

Check that gradients have been computed correctly.

In [14]:
print("\n" + "="*80)
print("Gradient Inspection")
print("="*80)

# Collect gradient statistics
grad_stats = []
total_grad_norm = 0.0
params_with_grad = 0
params_without_grad = 0

for name, param in model.named_parameters():
    if param.requires_grad:
        if param.grad is not None:
            grad = param.grad
            grad_norm = grad.norm().item()
            total_grad_norm += grad_norm ** 2
            params_with_grad += 1
            
            grad_stats.append({
                'name': name,
                'shape': list(param.shape),
                'grad_norm': grad_norm,
                'grad_min': grad.min().item(),
                'grad_max': grad.max().item(),
                'grad_mean': grad.mean().item(),
                'grad_std': grad.std().item(),
            })
        else:
            params_without_grad += 1

total_grad_norm = total_grad_norm ** 0.5

print(f"\nGradient statistics:")
print(f"  Parameters with gradients: {params_with_grad}")
print(f"  Parameters without gradients: {params_without_grad}")
print(f"  Total gradient norm: {total_grad_norm:.4f}")

# Show top 10 layers by gradient norm
print(f"\nTop 10 layers by gradient norm:")
sorted_stats = sorted(grad_stats, key=lambda x: x['grad_norm'], reverse=True)[:10]
for i, stat in enumerate(sorted_stats, 1):
    print(f"\n  {i}. {stat['name']}")
    print(f"     Shape: {stat['shape']}")
    print(f"     Grad norm: {stat['grad_norm']:.4f}")
    print(f"     Grad range: [{stat['grad_min']:.4f}, {stat['grad_max']:.4f}]")
    print(f"     Grad mean/std: {stat['grad_mean']:.4e} ± {stat['grad_std']:.4e}")


Gradient Inspection

Gradient statistics:
  Parameters with gradients: 76
  Parameters without gradients: 0
  Total gradient norm: 13.3000

Top 10 layers by gradient norm:

  1. model.transformer.h.0.mlp.c_proj.weight
     Shape: [1024, 256]
     Grad norm: 5.1902
     Grad range: [-0.0832, 0.0884]
     Grad mean/std: -1.0777e-06 ± 1.0137e-02

  2. model.transformer.h.1.mlp.c_proj.weight
     Shape: [1024, 256]
     Grad norm: 4.2787
     Grad range: [-0.0765, 0.0780]
     Grad mean/std: 8.2053e-08 ± 8.3569e-03

  3. model.transformer.h.0.attn.c_proj.bias
     Shape: [256]
     Grad norm: 3.7780
     Grad range: [-0.6555, 0.8131]
     Grad mean/std: 9.6352e-05 ± 2.3659e-01

  4. model.transformer.h.2.mlp.c_proj.weight
     Shape: [1024, 256]
     Grad norm: 3.4416
     Grad range: [-0.0804, 0.0777]
     Grad mean/std: 4.2394e-07 ± 6.7219e-03

  5. model.transformer.h.0.attn.c_proj.weight
     Shape: [256, 256]
     Grad norm: 3.2867
     Grad range: [-0.1309, 0.1601]
     Grad mean/st

## 9. Gradient Clipping (Optional)

Demonstrate gradient clipping as used in training.

In [15]:
print("\n" + "="*80)
print("Gradient Clipping")
print("="*80)

print(f"\nGradient norm before clipping: {total_grad_norm:.4f}")
print(f"Max gradient norm (from config): {cfg.training.max_grad_norm}")

# Clip gradients
torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.training.max_grad_norm)

# Recompute gradient norm after clipping
new_grad_norm = 0.0
for param in model.parameters():
    if param.grad is not None:
        new_grad_norm += param.grad.norm().item() ** 2
new_grad_norm = new_grad_norm ** 0.5

print(f"Gradient norm after clipping: {new_grad_norm:.4f}")

if total_grad_norm > cfg.training.max_grad_norm:
    print(f"\n✓ Gradients were clipped (reduced by {(1 - new_grad_norm/total_grad_norm)*100:.2f}%)")
else:
    print(f"\n✓ No clipping needed (gradient norm within limit)")


Gradient Clipping

Gradient norm before clipping: 13.3000
Max gradient norm (from config): 1.0
Gradient norm after clipping: 1.0000

✓ Gradients were clipped (reduced by 92.48%)


## 10. Optimizer Step (Simulated)

Demonstrate optimizer step without actually updating parameters.

In [16]:
print("\n" + "="*80)
print("Optimizer Setup")
print("="*80)

# Create optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
)

print(f"\nOptimizer: AdamW")
print(f"  Learning rate: {cfg.training.learning_rate}")
print(f"  Weight decay: {cfg.training.weight_decay}")
print(f"  Number of parameter groups: {len(optimizer.param_groups)}")

# Save a parameter value before update
first_param = next(model.parameters())
param_before = first_param.data.clone()
print(f"\nFirst parameter stats before update:")
print(f"  Mean: {param_before.mean().item():.6f}")
print(f"  Std: {param_before.std().item():.6f}")

# Perform optimizer step
print("\nPerforming optimizer step...")
optimizer.step()
print("Optimizer step completed!")

# Check parameter change
param_after = first_param.data
param_change = (param_after - param_before).abs().mean().item()
print(f"\nFirst parameter stats after update:")
print(f"  Mean: {param_after.mean().item():.6f}")
print(f"  Std: {param_after.std().item():.6f}")
print(f"  Average absolute change: {param_change:.8f}")

print(f"\n✓ Parameters have been updated!")


Optimizer Setup

Optimizer: AdamW
  Learning rate: 0.0001
  Weight decay: 0.01
  Number of parameter groups: 1

First parameter stats before update:
  Mean: 0.000029
  Std: 0.020017

Performing optimizer step...
Optimizer step completed!

First parameter stats after update:
  Mean: 0.000030
  Std: 0.020014
  Average absolute change: 0.00009995

✓ Parameters have been updated!


## 11. Summary

Summary of the test run.

In [17]:
print("\n" + "="*80)
print("TEST SUMMARY")
print("="*80)

print(f"\n✓ Configuration loaded from: {config_path.name}")
print(f"✓ Dataset loaded: {len(dataset)} samples")
print(f"✓ Model created: {params['total']:,} parameters")
print(f"✓ Forward pass successful")
print(f"✓ Backward pass successful")
print(f"✓ Gradients computed and clipped")
print(f"✓ Optimizer step completed")

print(f"\nMetrics from forward pass:")
print(f"  Loss: {outputs['loss'].item():.4f}")
print(f"  Accuracy: {outputs['accuracy'].item()*100:.2f}%")
print(f"  Top-5 Accuracy: {outputs['top5_accuracy'].item()*100:.2f}%")
print(f"  Perplexity: {outputs['perplexity'].item():.2f}")

print(f"\nModel is ready for training!")
print(f"\nNext steps:")
print(f"  1. Implement training loop over multiple epochs")
print(f"  2. Add validation evaluation")
print(f"  3. Add checkpointing and logging")
print(f"  4. Optionally enable WandB tracking")


TEST SUMMARY

✓ Configuration loaded from: trigo-gpt2.yaml
✓ Dataset loaded: 100 samples
✓ Model created: 5,329,664 parameters
✓ Forward pass successful
✓ Backward pass successful
✓ Gradients computed and clipped
✓ Optimizer step completed

Metrics from forward pass:
  Loss: 5.5731
  Accuracy: 3.48%
  Top-5 Accuracy: 5.50%
  Perplexity: 263.26

Model is ready for training!

Next steps:
  1. Implement training loop over multiple epochs
  2. Add validation evaluation
  3. Add checkpointing and logging
  4. Optionally enable WandB tracking
